# MVP Prototype: Home Assistant RAG

**Goal**: prove the RAG concept works end-to-end on a **small scale** before productionizing.

**Flow**: scrape a handful of doc pages -> chunk -> embed (local, fastembed) -> in-memory retrieval -> build prompt -> call LLM (Groq).

**Scope note**: this is a proof of concept only, so retrieval here is semantic (vector) search alone without lexical/keyword search or hybrid fusion yet. That's intentionally deferred to the production version once we move to Postgres+pgvector, where we can evaluate vector vs. text vs. hybrid properly.

Once this works and answers look reasonable, we refactor this logic into proper modules (`ingestion/`, `app/`) and swap in-memory search for Postgres+pgvector.

In [1]:
import os
import requests
import numpy as np
from bs4 import BeautifulSoup
from fastembed import TextEmbedding
from dotenv import load_dotenv
from groq import Groq
from minsearch import VectorSearch

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

## 1. Small doc sample

Just enough pages to sanity-check the pipeline. Full page list comes later in
`ingestion/sources.yaml` once we move to production phase.

In [2]:
SAMPLE_PAGES = [
    "https://www.home-assistant.io/docs/automation/templating/",
    "https://www.home-assistant.io/docs/automation/trigger/",
    "https://www.zigbee2mqtt.io/guide/getting-started/",
    "https://www.zigbee2mqtt.io/guide/usage/pairing_devices.html",
    "https://esphome.io/guides/getting_started_hassio.html",
]

## 2. Scrape + clean + chunk

In [ ]:
def fetch_page(url: str) -> str:
    resp = requests.get(url, timeout=20, headers={"User-Agent": "homeassistant-rag-mvp/0.1"})
    resp.raise_for_status()
    return resp.text


def html_to_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else ""
    main = soup.find("main") or soup.find("article") or soup.body or soup
    for tag in main.find_all(["nav", "script", "style", "footer", "header"]):
        tag.decompose()
    return title, main.get_text(separator="\n", strip=True)


def chunk_text(text: str, size: int = 800, overlap: int = 150) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return [c for c in chunks if len(c.strip()) > 50]

In [4]:
corpus = []  # list of {url, title, chunk_idx, content}

for url in SAMPLE_PAGES:
    html = fetch_page(url)
    title, text = html_to_text(html)
    for idx, chunk in enumerate(chunk_text(text)):
        corpus.append({"url": url, "title": title, "chunk_idx": idx, "content": chunk})

print(f"Total chunks: {len(corpus)}")
corpus[0]

Total chunks: 78


{'url': 'https://www.home-assistant.io/docs/automation/templating/',
 'title': 'Automation templates - Home Assistant',
 'chunk_idx': 0,
 'content': 'Automations support\ntemplating\nin the same way as scripts do. In addition to the\nHome Assistant template extensions\navailable to scripts, the\ntrigger\nand\nthis\ntemplate variables are available for automations.\nExample of variables used in templates:\n{{ this.name }}\nis the name of the automation executing from this trigger\n{{ trigger.platform }}\nis the type of trigger object, like\ncalendar\nAvailable state data\nThe template variable\nthis\nis an object that contains the\nstate\nof the automation at the moment of triggering the actions and can be used to evaluate\ntrigger_variables\ndeclared in the configuration of the active\ntrigger\nA trigger is a set of values or conditions of a platform that are defined to cause an automation to run.\n[Learn more]\n.\nState objects also contain context data which ca'}

## 3. Embed using fastembed/ONNX (local, no API cost)

In [5]:
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

texts = [c["content"] for c in corpus]
embeddings = np.array(list(embedder.embed(texts)))
embeddings.shape

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

(78, 384)

## 4. In-memory retrieval

Uses `minsearch.VectorSearch` to index the pre-computed embeddings and search by similarity. Still no database is used and will be swapped for Postgres+pgvector once we move to production phase (see plan in the main README).

In [6]:
vindex = VectorSearch(keyword_fields=[])
vindex.fit(embeddings, corpus)


def retrieve(query: str, top_k: int = 5):
    q_emb = list(embedder.embed([query]))[0]
    return vindex.search(q_emb, num_results=top_k)

## 5. Build prompt + call LLM (Groq)

In [7]:
PROMPT_TEMPLATE = """You are a helpful assistant answering questions about Home Assistant, \
Zigbee2MQTT, and ESPHome using the CONTEXT below. Only use the context; if the answer \
isn't there, say you don't know.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

client = groq_client


def answer(question: str, top_k: int = 5):
    chunks = retrieve(question, top_k=top_k)
    context = "\n\n".join(c["content"] for c in chunks)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return resp.choices[0].message.content, chunks

## 6. Smoke test

Sanity-check with a couple of questions we already know the answer to (from the sampled pages above).

In [8]:
for q in [
    "How do I template a sensor value in Home Assistant?",
    "How do I pair a new Zigbee device?",
]:
    resp, chunks = answer(q)
    print(f"Q: {q}\n")
    print(f"A: {resp}\n")
    print("Sources:", [c["url"] for c in chunks[:3]])
    print("-" * 100)

Q: How do I template a sensor value in Home Assistant?

A: You can template a sensor value in Home Assistant using the `value_template` option in the automation or sensor configuration. For example:

```yaml
sensor:
  - platform: template
    name: "My Sensor"
    value_template: "{{ states('sensor.my_sensor') | int }}"
```

In this example, the `value_template` is used to get the value of the `sensor.my_sensor` entity and convert it to an integer.

Alternatively, you can also use the `state` attribute to get the current state of the sensor:

```yaml
sensor:
  - platform: template
    name: "My Sensor"
    value_template: "{{ state_attr('sensor.my_sensor', 'value') }}"
```

This will get the current value of the `sensor.my_sensor` entity and use it as the value of the `My Sensor` sensor.

You can also use more complex templates to manipulate the value, such as:

```yaml
sensor:
  - platform: template
    name: "My Sensor"
    value_template: "{{ states('sensor.my_sensor') | int | round